# Module 7.1: LASSO and Elastic Net for Factor Selection

## Learning Objectives

By completing this notebook, you will:

1. Understand why high-dimensional regression requires regularization and how LASSO induces sparsity
2. Implement LASSO and elastic net regression for variable selection in factor models
3. Apply cross-validation to select optimal regularization parameters
4. Compare LASSO, elastic net, and standard PCA for forecasting with many predictors
5. Interpret coefficient paths and selected variables in economic contexts

## Prerequisites

- Principal Component Analysis (Module 1)
- Factor model estimation (Module 3)
- Basic understanding of linear regression
- Overfitting and bias-variance tradeoff

## Estimated Time: 90 minutes

---

## Setup and Imports

We'll use scikit-learn for LASSO/elastic net, numpy for numerical computation, and matplotlib for visualization.

In [ ]:
# Purpose: Import required libraries for high-dimensional regression
# Key Concept: LASSO and elastic net for sparse factor selection

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso, LassoCV, ElasticNet, ElasticNetCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. The High-Dimensional Problem

### Why OLS Fails with Many Predictors

When the number of predictors $N$ approaches or exceeds the number of observations $T$, ordinary least squares (OLS) becomes unstable or impossible:

$$\min_\beta \sum_{t=1}^T (y_t - x_t' \beta)^2$$

**Problems:**
- **Non-uniqueness**: If $N > T$, infinitely many solutions exist
- **Multicollinearity**: Correlated predictors lead to unstable estimates
- **Overfitting**: Model fits noise, poor out-of-sample performance

Let's demonstrate this with simulated data.

In [ ]:
# Purpose: Generate high-dimensional data with sparse true model
# Key Concept: Only a few predictors are truly relevant

def generate_sparse_data(T=100, N=200, n_relevant=10, noise_std=0.5, correlation=0.5):
    """
    Generate data where only n_relevant predictors affect target.
    
    Parameters
    ----------
    T : int
        Number of observations
    N : int
        Number of predictors (N > T for high-dimensional case)
    n_relevant : int
        Number of truly relevant predictors
    noise_std : float
        Standard deviation of noise
    correlation : float
        Correlation between predictors
    
    Returns
    -------
    X, y, true_beta, true_support
    """
    # Step 1: Create correlated predictor matrix
    # High correlation makes variable selection harder
    cov_matrix = correlation * np.ones((N, N)) + (1 - correlation) * np.eye(N)
    X = np.random.multivariate_normal(np.zeros(N), cov_matrix, size=T)
    
    # Step 2: Create sparse coefficient vector
    # Most coefficients are zero (sparse truth)
    true_support = np.random.choice(N, size=n_relevant, replace=False)
    true_beta = np.zeros(N)
    true_beta[true_support] = np.random.randn(n_relevant) * 2
    
    # Step 3: Generate target variable
    # y depends only on relevant predictors
    y = X @ true_beta + np.random.randn(T) * noise_std
    
    return X, y, true_beta, true_support

# Generate example data
T, N = 100, 200
n_relevant = 10
X, y, true_beta, true_support = generate_sparse_data(T=T, N=N, n_relevant=n_relevant)

print("="*70)
print("HIGH-DIMENSIONAL DATA CHARACTERISTICS")
print("="*70)
print(f"Observations (T): {T}")
print(f"Predictors (N): {N}")
print(f"Ratio N/T: {N/T:.2f}")
print(f"True relevant predictors: {n_relevant}")
print(f"True support indices: {sorted(true_support[:5])}... (first 5)")
print(f"Target variable mean: {y.mean():.3f}, std: {y.std():.3f}")
print("="*70)

### Visualizing the Sparse Truth

Let's visualize the true coefficient structure. Notice that most coefficients are exactly zero.

In [ ]:
# Purpose: Visualize sparse true coefficient structure
# Key Concept: True model has only few non-zero coefficients

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot 1: Bar plot of true coefficients
axes[0].bar(range(N), true_beta, alpha=0.6, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].scatter(true_support, true_beta[true_support], color='red', s=100, 
               marker='x', linewidth=3, label='True non-zero', zorder=5)
axes[0].set_xlabel('Predictor Index', fontsize=12)
axes[0].set_ylabel('Coefficient Value', fontsize=12)
axes[0].set_title('True Sparse Coefficient Structure', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Histogram of true coefficient magnitudes
axes[1].hist(np.abs(true_beta), bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero coefficient')
axes[1].set_xlabel('Absolute Coefficient Value', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Coefficient Magnitudes', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Sparsity level: {(true_beta == 0).sum()}/{N} coefficients are zero ({(true_beta == 0).sum()/N*100:.1f}%)")

## 2. LASSO Regression

### The LASSO Objective

LASSO (Least Absolute Shrinkage and Selection Operator) adds an L1 penalty to the least squares objective:

$$\hat{\beta}^{\text{LASSO}} = \arg\min_\beta \left\{ \frac{1}{2T} \sum_{t=1}^T (y_t - x_t' \beta)^2 + \lambda \sum_{j=1}^N |\beta_j| \right\}$$

**Key Properties:**
- **Sparsity**: The L1 penalty $|\beta_j|$ shrinks some coefficients exactly to zero
- **Variable Selection**: Automatically selects relevant predictors
- **Bias-Variance Tradeoff**: $\lambda$ controls the tradeoff between fit and complexity

**Why L1 Induces Sparsity:**
The L1 penalty has a "corner" at zero (not differentiable), which encourages exact zeros in the solution. Compare this to L2 (Ridge) which only shrinks coefficients toward zero but rarely makes them exactly zero.

In [ ]:
# Purpose: Fit LASSO regression with cross-validation for alpha selection
# Key Concept: Cross-validation chooses optimal regularization strength

# Step 1: Standardize predictors (essential for LASSO)
# LASSO penalty is scale-dependent
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Split into train and test sets
# Use temporal ordering for realistic evaluation
split = 80
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y[:split], y[split:]

# Step 3: Fit LASSO with cross-validation
# CV automatically selects optimal alpha (lambda)
print("Fitting LASSO with 5-fold cross-validation...")
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000, n_alphas=100)
lasso_cv.fit(X_train, y_train)

# Step 4: Extract results
optimal_alpha = lasso_cv.alpha_
selected_features = np.where(lasso_cv.coef_ != 0)[0]
n_selected = len(selected_features)

# Step 5: Evaluate on test set
y_pred_train = lasso_cv.predict(X_train)
y_pred_test = lasso_cv.predict(X_test)
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)

print("\n" + "="*70)
print("LASSO REGRESSION RESULTS")
print("="*70)
print(f"Optimal α (lambda): {optimal_alpha:.4f}")
print(f"Selected features: {n_selected} out of {N}")
print(f"True relevant features: {n_relevant}")
print(f"\nTraining MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R²: {test_r2:.4f}")

# Step 6: Check feature recovery
# How many true features did LASSO find?
recovered = len(set(selected_features) & set(true_support))
print(f"\nFeature Recovery:")
print(f"  True features recovered: {recovered}/{n_relevant} ({recovered/n_relevant*100:.1f}%)")
print(f"  False discoveries: {n_selected - recovered}")
print("="*70)

### Cross-Validation Results

Let's visualize how cross-validation error changes with the regularization parameter $\alpha$. The optimal $\alpha$ balances underfitting (too much regularization) and overfitting (too little regularization).

In [ ]:
# Purpose: Visualize cross-validation results for alpha selection
# Key Concept: Optimal alpha minimizes CV error

# Extract CV results
alphas = lasso_cv.alphas_
mse_path = lasso_cv.mse_path_
mse_mean = mse_path.mean(axis=1)
mse_std = mse_path.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))

# Plot mean CV error with standard deviation band
ax.plot(np.log10(alphas), mse_mean, 'b-', linewidth=2, label='Mean CV MSE')
ax.fill_between(np.log10(alphas), 
                mse_mean - mse_std, 
                mse_mean + mse_std,
                alpha=0.3, color='blue', label='±1 Std Dev')

# Mark optimal alpha
best_idx = np.argmin(mse_mean)
ax.axvline(np.log10(alphas[best_idx]), color='red', linestyle='--', 
          linewidth=2, label=f'Optimal α={alphas[best_idx]:.4f}')
ax.scatter(np.log10(alphas[best_idx]), mse_mean[best_idx], 
          color='red', s=100, zorder=5)

# 1-SE rule: largest alpha within 1 SE of minimum
se_threshold = mse_mean[best_idx] + mse_std[best_idx]
larger_alphas_mask = mse_mean <= se_threshold
if larger_alphas_mask.any():
    alpha_1se_idx = np.where(larger_alphas_mask)[0][-1]
    alpha_1se = alphas[alpha_1se_idx]
    ax.axvline(np.log10(alpha_1se), color='orange', linestyle=':', 
              linewidth=2, label=f'1-SE rule α={alpha_1se:.4f}')

ax.set_xlabel('log₁₀(α)', fontsize=12)
ax.set_ylabel('Cross-Validation MSE', fontsize=12)
ax.set_title('LASSO Cross-Validation Results', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### LASSO Regularization Path

The **regularization path** shows how coefficients evolve as we increase $\alpha$ from 0 (no regularization) to large values (heavy regularization). Notice how coefficients are progressively shrunk to zero.

In [ ]:
# Purpose: Compute and plot LASSO regularization path
# Key Concept: Coefficients shrink to zero as alpha increases

# Compute path across range of alphas
n_alphas = 100
alphas_path = np.logspace(-4, 1, n_alphas)
coefs = []

for alpha in alphas_path:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train, y_train)
    coefs.append(lasso.coef_)

coefs = np.array(coefs)

# Plot regularization path
fig, ax = plt.subplots(figsize=(12, 6))

# Plot all coefficient paths
for i in range(N):
    if i in true_support:
        # Highlight true non-zero coefficients in red
        ax.plot(np.log10(alphas_path), coefs[:, i], 'r-', linewidth=2, alpha=0.7)
    else:
        # Plot other coefficients in blue
        ax.plot(np.log10(alphas_path), coefs[:, i], 'b-', linewidth=1, alpha=0.3)

# Mark selected alpha
ax.axvline(np.log10(optimal_alpha), color='green', linestyle='--', 
          linewidth=2, label=f'Selected α={optimal_alpha:.4f}')

# Create custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='r', linewidth=2, label='True non-zero coef.'),
    Line2D([0], [0], color='b', linewidth=1, label='True zero coef.'),
    Line2D([0], [0], color='g', linestyle='--', linewidth=2, label='Selected α')
]
ax.legend(handles=legend_elements, fontsize=10)

ax.set_xlabel('log₁₀(α)', fontsize=12)
ax.set_ylabel('Coefficient Value', fontsize=12)
ax.set_title('LASSO Regularization Path', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nNote: Red lines show true relevant predictors.")
print(f"As α increases, coefficients are progressively set to zero.")

## 3. Elastic Net Regression

### Motivation for Elastic Net

LASSO has limitations when predictors are highly correlated:
- Arbitrarily selects one predictor from a correlated group
- Can select at most $T$ variables (when $N > T$)
- Unstable selection

**Elastic Net** combines L1 and L2 penalties:

$$\hat{\beta}^{\text{EN}} = \arg\min_\beta \left\{ \frac{1}{2T} \sum_{t=1}^T (y_t - x_t' \beta)^2 + \lambda \left[ \alpha \|\beta\|_1 + \frac{1-\alpha}{2} \|\beta\|_2^2 \right] \right\}$$

where $\alpha \in [0,1]$ controls the mixing:
- $\alpha = 1$: Pure LASSO
- $\alpha = 0$: Pure Ridge
- $\alpha \in (0,1)$: Elastic Net

In [ ]:
# Purpose: Fit Elastic Net with cross-validation
# Key Concept: Elastic net handles correlated predictors better than LASSO

print("Fitting Elastic Net with 5-fold cross-validation...")
enet_cv = ElasticNetCV(cv=5, l1_ratio=0.5, random_state=42, max_iter=10000, n_alphas=100)
enet_cv.fit(X_train, y_train)

# Extract results
enet_alpha = enet_cv.alpha_
enet_selected = np.where(enet_cv.coef_ != 0)[0]
enet_n_selected = len(enet_selected)

# Evaluate
enet_pred_train = enet_cv.predict(X_train)
enet_pred_test = enet_cv.predict(X_test)
enet_train_mse = mean_squared_error(y_train, enet_pred_train)
enet_test_mse = mean_squared_error(y_test, enet_pred_test)
enet_test_r2 = r2_score(y_test, enet_pred_test)

# Feature recovery
enet_recovered = len(set(enet_selected) & set(true_support))

print("\n" + "="*70)
print("ELASTIC NET RESULTS (l1_ratio=0.5)")
print("="*70)
print(f"Optimal α: {enet_alpha:.4f}")
print(f"Selected features: {enet_n_selected} out of {N}")
print(f"\nTraining MSE: {enet_train_mse:.4f}")
print(f"Test MSE: {enet_test_mse:.4f}")
print(f"Test R²: {enet_test_r2:.4f}")
print(f"\nFeature Recovery:")
print(f"  True features recovered: {enet_recovered}/{n_relevant} ({enet_recovered/n_relevant*100:.1f}%)")
print(f"  False discoveries: {enet_n_selected - enet_recovered}")
print("="*70)

## 4. Comparing Methods

Let's compare LASSO, Elastic Net, Ridge, and standard PCA on our high-dimensional problem.

In [ ]:
# Purpose: Compare multiple regularization methods
# Key Concept: Different methods have different strengths

# Fit additional methods for comparison

# Ridge regression (L2 penalty only)
print("Fitting Ridge regression...")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_test_mse = mean_squared_error(y_test, ridge.predict(X_test))
ridge_n_selected = N  # Ridge doesn't zero out coefficients

# PCA + regression
print("Fitting PCA factor model...")
n_components = min(10, X_train.shape[0] - 1)  # Use 10 factors
pca = PCA(n_components=n_components)
F_train = pca.fit_transform(X_train)
F_test = pca.transform(X_test)

from sklearn.linear_model import LinearRegression
ols = LinearRegression()
ols.fit(F_train, y_train)
pca_test_mse = mean_squared_error(y_test, ols.predict(F_test))
pca_n_selected = n_components  # PCA uses all components

# Summary comparison
results_df = pd.DataFrame({
    'Method': ['LASSO', 'Elastic Net', 'Ridge', 'PCA'],
    'Test MSE': [test_mse, enet_test_mse, ridge_test_mse, pca_test_mse],
    'Variables Selected': [n_selected, enet_n_selected, ridge_n_selected, pca_n_selected],
    'True Features Found': [recovered, enet_recovered, np.nan, np.nan]
})

print("\n" + "="*70)
print("METHOD COMPARISON")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Determine best method
best_method = results_df.loc[results_df['Test MSE'].idxmin(), 'Method']
print(f"\nBest method by test MSE: {best_method}")

### Visualizing Method Comparison

In [ ]:
# Purpose: Visualize comparison of different methods
# Key Concept: Trade-off between sparsity and prediction accuracy

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Test MSE comparison
methods = results_df['Method']
test_mses = results_df['Test MSE']
colors = ['steelblue', 'seagreen', 'coral', 'purple']

bars = axes[0].bar(methods, test_mses, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Test MSE', fontsize=12)
axes[0].set_title('Out-of-Sample Prediction Error', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Highlight best method
best_idx = test_mses.argmin()
bars[best_idx].set_edgecolor('red')
bars[best_idx].set_linewidth(3)

# Plot 2: Number of variables selected
n_vars = results_df['Variables Selected']
axes[1].bar(methods, n_vars, color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(n_relevant, color='red', linestyle='--', linewidth=2, label=f'True: {n_relevant}')
axes[1].set_ylabel('Variables Selected', fontsize=12)
axes[1].set_title('Model Sparsity', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Coefficient comparison for LASSO and Elastic Net
coef_comparison = pd.DataFrame({
    'LASSO': lasso_cv.coef_,
    'Elastic Net': enet_cv.coef_,
    'True': true_beta
})

x_pos = np.arange(N)
width = 0.35
axes[2].bar(x_pos - width/2, np.abs(lasso_cv.coef_), width, 
           label='LASSO', alpha=0.6, color='steelblue')
axes[2].bar(x_pos + width/2, np.abs(enet_cv.coef_), width, 
           label='Elastic Net', alpha=0.6, color='seagreen')
axes[2].scatter(true_support, np.abs(true_beta[true_support]), 
               color='red', s=50, marker='x', linewidth=2, 
               label='True non-zero', zorder=5)
axes[2].set_xlabel('Predictor Index', fontsize=12)
axes[2].set_ylabel('Absolute Coefficient', fontsize=12)
axes[2].set_title('Coefficient Comparison', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].set_xlim(-5, min(50, N))  # Show first 50 predictors for clarity
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Exercises

### Exercise 5.1: Understanding the Soft-Thresholding Operator

The soft-thresholding operator is key to LASSO's sparsity:

$$S(z, \gamma) = \begin{cases}
z - \gamma & \text{if } z > \gamma \\
0 & \text{if } |z| \leq \gamma \\
z + \gamma & \text{if } z < -\gamma
\end{cases}$$

**Task:** Implement the soft-thresholding operator and visualize how it works.

In [ ]:
# Exercise 5.1: Implement soft-thresholding operator

def soft_threshold(z, gamma):
    """
    Soft-thresholding operator.
    
    Parameters
    ----------
    z : float or array
        Input value(s)
    gamma : float
        Threshold parameter
    
    Returns
    -------
    thresholded : float or array
        Soft-thresholded output
    """
    # YOUR CODE HERE
    # Hint: Use np.sign, np.maximum, or conditional logic
    # ---------------
    
    result = None  # Replace with your implementation
    
    return result

# Test your implementation
z_values = np.linspace(-5, 5, 100)
gamma_values = [0.5, 1.0, 2.0]

plt.figure(figsize=(10, 6))
plt.plot(z_values, z_values, 'k--', label='Identity (no threshold)', linewidth=2)

for gamma in gamma_values:
    thresholded = soft_threshold(z_values, gamma)
    plt.plot(z_values, thresholded, label=f'γ = {gamma}', linewidth=2)

plt.xlabel('Input (z)', fontsize=12)
plt.ylabel('Output S(z, γ)', fontsize=12)
plt.title('Soft-Thresholding Operator', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# AUTO-GRADED TESTS
def test_exercise_5_1():
    assert soft_threshold is not None, "Implement soft_threshold function"
    
    # Test single values
    assert np.isclose(soft_threshold(2.0, 0.5), 1.5), "soft_threshold(2.0, 0.5) should be 1.5"
    assert np.isclose(soft_threshold(-2.0, 0.5), -1.5), "soft_threshold(-2.0, 0.5) should be -1.5"
    assert np.isclose(soft_threshold(0.3, 0.5), 0.0), "soft_threshold(0.3, 0.5) should be 0.0"
    
    # Test array
    z_test = np.array([-3, -1, 0, 1, 3])
    expected = np.array([-2, 0, 0, 0, 2])
    result = soft_threshold(z_test, 1.0)
    assert np.allclose(result, expected), f"Expected {expected}, got {result}"
    
    print("✅ Exercise 5.1 passed!")

test_exercise_5_1()

### Exercise 5.2: Comparing Different L1 Ratios

**Task:** Fit elastic net models with different `l1_ratio` values (0.1, 0.5, 0.9) and compare their performance and sparsity.

In [ ]:
# Exercise 5.2: Compare elastic net with different l1_ratios

l1_ratios = [0.1, 0.5, 0.9]

# YOUR CODE HERE
# ---------------
# For each l1_ratio:
#   1. Fit ElasticNetCV
#   2. Compute test MSE
#   3. Count selected features
#   4. Store results in a list or dictionary

results = []  # Replace with your implementation

# Create comparison DataFrame
comparison_df = None  # YOUR CODE: Create DataFrame from results

# Plot results
# YOUR CODE: Create bar plots comparing test MSE and sparsity

# AUTO-GRADED TESTS
def test_exercise_5_2():
    assert comparison_df is not None, "Create comparison_df with results"
    assert len(comparison_df) == 3, "Should have 3 rows (one per l1_ratio)"
    assert 'l1_ratio' in comparison_df.columns, "Should have 'l1_ratio' column"
    assert 'Test MSE' in comparison_df.columns, "Should have 'Test MSE' column"
    assert 'N Selected' in comparison_df.columns, "Should have 'N Selected' column"
    
    print("✅ Exercise 5.2 passed!")

test_exercise_5_2()

### Exercise 5.3: Stability of Variable Selection

**Task:** Assess stability of LASSO variable selection using bootstrap resampling. How often does LASSO select each variable across 100 bootstrap samples?

In [ ]:
# Exercise 5.3: Bootstrap stability analysis

n_bootstrap = 100
selection_counts = np.zeros(N)

# YOUR CODE HERE
# ---------------
# For each bootstrap iteration:
#   1. Resample training data with replacement
#   2. Fit LASSO on bootstrap sample
#   3. Record which variables are selected
#   4. Accumulate counts in selection_counts

# Calculate selection frequency
selection_freq = None  # YOUR CODE: selection_counts / n_bootstrap

# Visualize stability
# YOUR CODE: Create bar plot showing selection frequency for each variable
# Highlight true non-zero variables

# AUTO-GRADED TESTS
def test_exercise_5_3():
    assert selection_freq is not None, "Calculate selection_freq"
    assert len(selection_freq) == N, f"selection_freq should have length {N}"
    assert np.all((selection_freq >= 0) & (selection_freq <= 1)), "Frequencies should be in [0, 1]"
    assert selection_counts.sum() > 0, "Run bootstrap procedure to populate selection_counts"
    
    # Check that true features are selected more frequently
    true_freq_mean = selection_freq[true_support].mean()
    false_freq_mean = selection_freq[np.setdiff1d(np.arange(N), true_support)].mean()
    assert true_freq_mean > false_freq_mean, "True features should be selected more frequently"
    
    print("✅ Exercise 5.3 passed!")
    print(f"\nTrue features selected on average: {true_freq_mean*100:.1f}% of bootstrap samples")
    print(f"False features selected on average: {false_freq_mean*100:.1f}% of bootstrap samples")

test_exercise_5_3()

## 6. Summary

### Key Takeaways

1. **High-dimensional regression** (N ≥ T) requires regularization to avoid overfitting and enable variable selection

2. **LASSO uses L1 penalty** ($\lambda \sum |\beta_j|$) which induces exact sparsity by setting coefficients to zero

3. **Elastic Net combines L1 and L2** penalties, handling correlated predictors better than pure LASSO

4. **Cross-validation** is essential for selecting optimal regularization parameters

5. **Regularization path** shows how coefficients shrink to zero as penalty increases

6. **Trade-offs exist** between sparsity (interpretability) and prediction accuracy

### When to Use Each Method

- **LASSO**: When true model is sparse, predictors not highly correlated
- **Elastic Net**: When predictors are correlated (factor models), want stability
- **Ridge**: When many small effects, no sparsity needed
- **PCA**: When dimension reduction without selection is goal

### Next Steps

In the next notebook, we'll explore **targeted predictors** (Bai-Ng method), which combines variable screening with factor extraction to improve forecasting performance.

## Additional Resources

**Key Papers:**
- Tibshirani, R. (1996). "Regression Shrinkage and Selection via the Lasso." *JRSS-B*
- Zou, H. & Hastie, T. (2005). "Regularization and Variable Selection via the Elastic Net." *JRSS-B*

**Books:**
- Hastie, T., Tibshirani, R., & Wainwright, M. (2015). *Statistical Learning with Sparsity*

**Software:**
- scikit-learn documentation: [Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)